# GSV-Math Backend — Fast One-Click Colab GPU Server

### Instructions:
1. In the top menu: **Runtime → Change runtime type → select T4 GPU → Save**.
2. Paste your **ngrok token** in **Cell 1**.
3. Click **Runtime → Run all** (`Ctrl + F9`).
4. Copy the public `https://...ngrok-free.app` URL printed in **Cell 4** and paste it into Vercel!


In [ ]:
# ============================================================
# CELL 1: Paste your ngrok authtoken here
# Get it for free at: https://dashboard.ngrok.com/get-started/your-authtoken
# ============================================================
NGROK_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"
API_KEY = "dev-secret-key"

print("✅ Configuration set! Now run the next cells.")


In [ ]:
# ============================================================
# CELL 2: Install Dependencies (~30 seconds, 0 C++ compilation)
# ============================================================
!pip install -q -U transformers accelerate bitsandbytes peft fastapi uvicorn pyngrok 'Pillow>=10.4.0,<11.0.0' httpx sympy
print("✅ Dependencies installed successfully!")


In [ ]:
# ============================================================
# CELL 3: Load Qwen2.5-VL-7B in 4-bit on T4 GPU (~1 minute)
# Uses the SAME quantization config as training to avoid skew.
# ============================================================
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel

# Training-matched pixel budget
MAX_PIXELS = 156800   # from project_notebooks/Qwen_2.5v_finetuning.ipynb
MIN_PIXELS = 3136     # 56*56

print("Loading Qwen2.5-VL-7B on T4 GPU with training-matched 4-bit config...")

# Match training quantization exactly
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Pin processor pixel budget to match training
processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)

# Apply LoRA adapter
print("Applying LoRA adapter...")
model = PeftModel.from_pretrained(model, "Shabuuuuuuuuuuu/GSV-Math-Qwen2.5-VL-7B-Expert")
model.eval()

print("✅ Model + LoRA loaded on T4 GPU successfully!")


In [ ]:
# ============================================================
# CELL 4: Start FastAPI Server + ngrok Public Tunnel
# Fixes: extraction bug, system prompt, pixel budget, decode
# ============================================================
import os, io, re, base64, threading, httpx, uvicorn, unicodedata
from PIL import Image
from fastapi import FastAPI, Request, Depends, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.security.api_key import APIKeyHeader
from pyngrok import ngrok, conf

# ── Training-matched constants ──
PIL_MAX_SIDE = 396  # sqrt(156800) ≈ 396
SYSTEM_PROMPT = (
    "You are a math problem solver. Look at the image carefully, "
    "read the question, and solve it step by step. "
    "Show your work and end with: The answer is <your answer>."
)

# ── Unicode canonicalization ──
_UNICODE_MAP = {
    '√': 'sqrt', '×': '*', '÷': '/', 'π': 'pi',
    '²': '^2', '³': '^3', '⁰': '^0', '¹': '^1',
    '⁴': '^4', '⁵': '^5', '⁶': '^6', '⁷': '^7',
    '⁸': '^8', '⁹': '^9', '–': '-', '—': '-',
    '−': '-', '½': '1/2', '¼': '1/4', '¾': '3/4',
    '⅓': '1/3', '⅔': '2/3', '°': '',
}

_UNITS = sorted([
    'square units', 'sq units', 'cubic units', 'cu units',
    'units', 'unit', 'centimeters', 'centimeter', 'cm²', 'cm2', 'cm',
    'meters', 'meter', 'mm', 'm²', 'm2', 'm',
    'inches', 'inch', 'in', 'feet', 'foot', 'ft',
    'degrees', 'degree', 'deg', '°', 'radians', 'radian', 'rad',
    'percent', '%', 'dollars', 'dollar', '$',
], key=len, reverse=True)

# ── Fixed extraction patterns (audit D.15) ──
FINAL_PATTERNS = [
    r'\boxed{([^}]*)}',
    r'(?:[Ss]o\s+)?[Tt]he\s+answer\s+is\s*[:\-]?\s*(.{1,80})',
    r'[Tt]herefore[,\s]+(?:the\s+)?(?:answer|value|result)\s+is\s*[:\-]?\s*(.{1,80})',
    r'[Ff]inal\s*[Aa]nswer\s*[:\-=]\s*(.{1,80})',
    r'[Ff]inal\s*[Aa]nswer\s+is\s*[:\-]?\s*(.{1,80})',
    r'=\s*(\S+)\s*$',
]

def _canonicalize_unicode(text):
    for uc, repl in _UNICODE_MAP.items():
        text = text.replace(uc, repl)
    return unicodedata.normalize('NFKD', text)

def extract_answer(text):
    if not isinstance(text, str): return str(text)
    for p in FINAL_PATTERNS:
        m = list(re.finditer(p, text, re.IGNORECASE | re.DOTALL))
        if m:
            ans = m[-1].group(1).strip()
            ans = re.split(r'[.\n]', ans)[0].strip()
            if ans: return ans
    idx = text.lower().rfind("answer is")
    if idx != -1:
        ans = text[idx+9:].strip()
        ans = re.split(r'[.\n]', ans)[0].strip()
        ans = ans.replace(":", "").strip()
        if ans: return ans
    words = text.split()
    return words[-1].rstrip('.,;:!?') if words else text

def normalize_answer(ans):
    if not ans: return ""
    ans = _canonicalize_unicode(ans).strip().lower()
    ans = re.sub(r'^[a-z]\s*=\s*', '', ans)
    for unit in _UNITS:
        if ans.endswith(unit):
            ans = ans[:-len(unit)].strip()
            break
    ans = ans.rstrip('.,;:!?')
    try:
        f = float(ans)
        return str(int(f)) if f.is_integer() else str(f)
    except ValueError:
        return ans

# ── FastAPI app ──
app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)

async def verify_api_key(api_key: str = Depends(api_key_header)):
    if api_key != API_KEY:
        raise HTTPException(status_code=403, detail="Invalid API Key")

@app.post("/", dependencies=[Depends(verify_api_key)])
async def solve(request: Request):
    try:
        data = await request.json()
        image_b64 = data.get("image_base64")
        image_url = data.get("image_url")
        question = data.get("question", "")
        if not (image_b64 or image_url) or not question:
            return {"error": "Missing image or question in payload"}
        if image_url and not image_b64:
            async with httpx.AsyncClient(timeout=10) as client:
                resp = await client.get(image_url)
                image_b64 = base64.b64encode(resp.content).decode()
        image_bytes = base64.b64decode(image_b64)
        img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        # Pre-resize to match training resolution
        if max(img.size) > PIL_MAX_SIDE:
            img.thumbnail((PIL_MAX_SIDE, PIL_MAX_SIDE))
        # Build messages with system prompt
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": question}]}
        ]
        text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text_prompt], images=[img], return_tensors="pt").to("cuda")
        prompt_len = inputs['input_ids'].shape[1]  # for slicing off echoed prompt
        with torch.no_grad():
            out_ids = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True)
        # Decode only GENERATED tokens (skip echoed prompt)
        generated_ids = out_ids[0][prompt_len:]
        raw_response = processor.decode(generated_ids, skip_special_tokens=True)
        extracted_ans = normalize_answer(extract_answer(raw_response))
        return {
            "answer": extracted_ans,
            "reasoning": raw_response,
            "vote_distribution": {extracted_ans: 1.0},
            "owl_grounding_score": None,
            "clip_alignment_score": None,
            "symbolic_check_passed": None,
            "note": "Running live on Colab T4 GPU"
        }
    except Exception as e:
        return {"error": str(e)}

@app.get("/health")
def health(): return {"status": "ok", "message": "Colab GPU server is alive!"}

def start_uvicorn(): uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")
threading.Thread(target=start_uvicorn, daemon=True).start()

import time; time.sleep(2)
conf.get_default().auth_token = NGROK_TOKEN
tunnel = ngrok.connect(8000, "http")
PUBLIC_URL = tunnel.public_url

print("=" * 60)
print("🎉 YOUR VERCEL BACKEND IS LIVE!")
print("=" * 60)
print(f"\n👉 Public URL: {PUBLIC_URL}\n")
print("Go to Vercel → Settings → Environment Variables:")
print(f"  NEXT_PUBLIC_MODAL_BACKEND_URL = {PUBLIC_URL}")
print("=" * 60)


In [ ]:
# ============================================================
# CELL 5: Keep-Alive Loop (Keeps Session Running)
# ============================================================
import time, requests
print("Keep-alive active. Leave this cell running in the background!")
count = 0
while True:
    try:
        r = requests.get("http://localhost:8000/health", timeout=5)
        count += 1
        if count % 12 == 0:
            print(f"  🟢 Backend healthy | Uptime: {count * 5}s | URL: {PUBLIC_URL}")
    except:
        pass
    time.sleep(5)
